In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

# FIX: reduce GPU memory fragmentation (this is exactly what the
# "OutOfMemoryError...try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True"
# message suggests). Must be set before any CUDA context is created, so it
# goes at the very top, before torch is imported anywhere.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

print(os.path.exists("/content/drive"))
print(os.path.exists("/content/drive/MyDrive"))

# ======================================================================
# EVALUATION PIPELINE: No-RAG vs RAG vs RAC
# ======================================================================
# RAC (Retrieval-Augmented Classification) follows Chang et al.,
# "Retrieval Augmented Classification for Confidential Documents"
# (ICONI 2025). The paper's key finding was NOT that reference passages
# alone help most — plain retrieval-without-examples (their "0-shot RAG")
# already got them to ~93% accuracy. The big additional jump (to ~96%
# accuracy / ~94% F1) came from retrieving SOLVED, ANSWER-BALANCED
# EXEMPLARS (similar question + its correct label) and using them as
# few-shot demonstrations, with k=3 being the best-performing setting on
# the imbalanced/original split (Fig. 3, Sec. 4.3-4.4).
#
# So this script keeps your existing No-RAG and RAG (passage-only) arms
# unchanged in spirit, and adds a third arm, RAC, that:
#   1. Retrieves the k most similar SOLVED questions from a labeled
#      exemplar pool (dense retrieval + cross-encoder rerank, same
#      embedder/reranker you already load).
#   2. Greedily re-orders the reranked candidates to diversify the
#      correct-answer letters among the chosen exemplars (mirrors the
#      paper's "one example per class" balanced few-shot selection,
#      Sec. 3.2) so the model isn't just pattern-matching a repeated
#      letter.
#   3. Builds a few-shot prompt from those worked examples (optionally
#      combined with the RAG passages) and asks the model to answer
#      the target question in the same structured "Answer: X" format.
#
# CHANGES FROM YOUR ORIGINAL SCRIPT:
#   - Added Section 7b: exemplar pool loader + RAC retrieval/prompting
#   - Added RAC arm to the evaluation loop, RAC config block
#   - Fixed divide-by-zero risk in BM25 min-max normalization
#   - Fixed pipeline output not always being a plain string
#   - Added explicit self-exclusion guard so RAC can never retrieve the
#     row currently being scored as its own exemplar (label leakage)
#   - Minor cleanup: helper functions instead of duplicated eval code,
#     3-way summary + 3-way error analysis at the end
#   - Added 3 more subjects (speech_pathologist, biomedical_engineer,
#     occupational_therapist): SYSTEM_PROMPT -> SYSTEM_PROMPTS dict,
#     CATEGORY_HINTS extended, formatter functions now take `subject`
# ======================================================================
# INSTRUCTIONS:
#   1. First run the KB build script to create the passage KB (used by
#      RAG and, optionally, RAC).
#   2. Upload {subject}_test.tsv for every subject in SUBJECTS (required).
#      If you also have {subject}_train.tsv or {subject}_dev.tsv under
#      EVAL_DATA_PATH/train or EVAL_DATA_PATH/dev, RAC will use that as
#      its exemplar pool. If neither exists, RAC automatically falls
#      back to leave-one-out retrieval over the test set itself (each
#      query excludes its own row, so there is no label leakage).
#   3. Run this entire cell in Colab.
#   4. For T4 GPU (free Colab): uses 4-bit quantization, fits in 16GB.
#   5. For A100 (Colab Pro): set USE_4BIT = False for better quality.
# ======================================================================

# -------------------------
# 1) Install
# -------------------------
import subprocess, sys
packages = [
    "transformers>=4.40.0",
    "accelerate>=0.28.0",
    "bitsandbytes>=0.43.0",
    "sentence-transformers>=2.2.2",
    "huggingface_hub>=0.20.3",
    "faiss-cpu",
    "rank-bm25",
    "scikit-learn",
    "thefuzz",
    "python-Levenshtein",
    "tqdm",
]
for p in packages:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])
    except Exception as e:
        print(f"[WARN] {p}: {e}")

# -------------------------
# 2) Imports
# -------------------------
import os
import re
import time
import pickle
import warnings
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from thefuzz import process
import faiss
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from transformers.trainer_utils import set_seed
from sentence_transformers import SentenceTransformer, CrossEncoder

warnings.filterwarnings("ignore")
# Silence the repeated "Both `max_new_tokens` and `max_length` seem to have
# been set" notice and other routine transformers logging noise.
transformers.logging.set_verbosity_error()
import logging as _logging
_logging.getLogger("transformers").setLevel(_logging.ERROR)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# -------------------------
# 3) Config
# -------------------------
# NOTE: each string below must match EXACTLY across four places:
#   1. this SUBJECTS list
#   2. the SYSTEM_PROMPTS dict keys (Section 6)
#   3. the CATEGORY_HINTS dict keys (Section 8)
#   4. the {subject}_test.tsv / {subject}_train.tsv / {subject}_dev.tsv filenames
SUBJECTS = [
     "clinical_psychologist",
      "occupational_therapist",
      "biomedical_engineer",
      "speech_pathologist",
]

CHOICES = ["A", "B", "C", "D"]

class Config:
    # ===== MODEL CONFIG =====
    LLM_MODEL = "Qwen/Qwen2.5-7B-Instruct"
    USE_4BIT = True

    # ===== RETRIEVAL CONFIG (RAG passages) =====
    EMBEDDER_MODEL = "BAAI/bge-base-en-v1.5"
    BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
    RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
    KB_DIR = "/content/drive/MyDrive/KB_complete"
    EVAL_DATA_PATH = "/content/drive/MyDrive/eval_data"
    OUTPUT_DIR = "/content/drive/MyDrive/results_improved"

    RETRIEVE_K = 30
    FINAL_K = 3
    BM25_TOP = 300
    SEM_WEIGHT = 0.7
    CATEGORY_BONUS = 0.12
    RERANK_THRESHOLD = 0.0
    MAX_PASSAGE_WORDS = 150
    MAX_NEW_TOKENS = 512
    SEED = 1234

    # ===== RAC CONFIG (retrieved worked examples) =====
    RUN_RAC = True
    RAC_EXEMPLAR_RETRIEVE_K = 20     # candidate pool size before rerank
    RAC_SHOTS = 3                    # paper's best setting on imbalanced data (Sec 4.4)
    # FIX: the cross-encoder reranker (ms-marco-MiniLM) is trained for
    # query-vs-passage relevance, not question-vs-question similarity, and
    # its raw logit scores are commonly negative even for genuinely related
    # question pairs. A hard threshold of 0.0 was silently discarding almost
    # every exemplar candidate, making RAC collapse to plain RAG for most
    # questions. Default is now None: always keep the top RAC_SHOTS by rank,
    # regardless of absolute score. Set to a float only if you've checked
    # (via the sanity-check script) what a reasonable cutoff looks like for
    # your data.
    RAC_RERANK_THRESHOLD = None
    RAC_COMBINE_WITH_PASSAGES = True  # add RAG passages on top of exemplars
    RAC_INCLUDE_RATIONALE = True      # show exp0/exp1 rationale in exemplars if present
    RAC_MAX_EXEMPLAR_WORDS = 120
    # FIX: the paper's "balanced few-shot" (one example per class) makes
    # sense for their task because the class label (Secret/Confidential/
    # Unclassified) is a real, meaningful category. For MCQ, the answer
    # letter (A/B/C/D) is arbitrary -- it's just where the correct choice
    # happens to sit -- so forcing letter diversity has no semantic value
    # and was observed to discard more-relevant exemplars in favor of
    # irrelevant ones just to hit a different letter. Default is now off:
    # exemplars are chosen purely by relevance (reranker score).
    RAC_BALANCE_ANSWERS = False

set_seed(Config.SEED)
os.makedirs(Config.OUTPUT_DIR, exist_ok=True)

# -------------------------
# 4) BM25 Tokenizer (MUST MATCH KB BUILD)
# -------------------------
STOPWORDS = frozenset({
    'the', 'a', 'an', 'is', 'are', 'was', 'were', 'be', 'been', 'being',
    'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'shall',
    'should', 'may', 'might', 'can', 'could', 'and', 'but', 'or', 'nor',
    'not', 'no', 'so', 'if', 'then', 'than', 'that', 'this', 'these',
    'those', 'it', 'its', 'of', 'in', 'on', 'at', 'to', 'for', 'with',
    'by', 'from', 'as', 'into', 'about', 'between', 'through', 'after',
    'before', 'during', 'without', 'within', 'also', 'more', 'most',
    'very', 'such', 'each', 'both', 'all', 'any', 'some', 'other',
    'which', 'what', 'when', 'where', 'how', 'who', 'whom', 'whose',
    'there', 'here', 'just', 'only', 'own', 'same', 'our', 'your',
    'their', 'my', 'his', 'her', 'we', 'they', 'you', 'he', 'she',
    'me', 'him', 'us', 'them', 'i', 'am',
})

def tokenize_for_bm25(text):
    """Must match the tokenizer used in KB build."""
    tokens = re.findall(r'[a-z0-9]+', text.lower())
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]

# -------------------------
# 5) Answer Extraction
# -------------------------
def process_before_extraction(gen, choice_dict):
    for key, val in sorted(choice_dict.items(), key=lambda x: len(str(x[1])), reverse=True):
        pattern = re.compile(re.escape(str(val).rstrip(".")), re.IGNORECASE)
        gen = pattern.sub(key, gen)
    return gen

def extract_choice(gen, choice_list):
    res = re.search(
        r"(?:(?:[Cc]hoose)|(?:(?:[Aa]nswer|[Cc]hoice)(?![^ABCD]{0,20}?(?:n't|not))[^ABCD]{0,10}?\b(?:|is|:|be))\b)[^ABCD]{0,20}?\b(A|B|C|D)\b",
        gen,
    )
    if res is None:
        res = re.search(
            r"\b(A|B|C|D)\b(?![^ABCD]{0,8}?(?:n't|not)[^ABCD]{0,5}?(?:correct|right))[^ABCD]{0,10}?\b(?:correct|right)\b",
            gen,
        )
    if res is None:
        res = re.search(r"^(A|B|C|D)(?:\.|,|:|$)", gen)
    if res is None:
        res = re.search(r"(?<![a-zA-Z])(A|B|C|D)(?![a-zA-Z=])", gen)
    if res is None:
        best = process.extractOne(gen, choice_list)
        return CHOICES[choice_list.index(best[0])] if best else "A"
    return res.group(1)

def extract_answer(response, row):
    """Try structured format first, then fall back to regex/fuzzy matching."""
    if not isinstance(response, str):
        response = str(response)

    m = re.search(r'[Aa]nswer\s*:\s*\**\s*([A-D])\b', response)
    if m:
        return m.group(1)
    m = re.search(r'[Tt]he\s+answer\s+is\s+\**\s*([A-D])\b', response)
    if m:
        return m.group(1)
    m = re.search(r'correct\s+answer\s+is\s+\**\s*([A-D])\b', response)
    if m:
        return m.group(1)
    m = re.search(r'\*\*([A-D])\*\*', response)
    if m:
        return m.group(1)
    last_lines = response.strip().split('\n')
    for line in reversed(last_lines[-3:]):
        m = re.search(r'\b([A-D])\b', line.strip())
        if m and len(line.strip()) < 50:
            return m.group(1)

    gen = process_before_extraction(response, {c: row[c] for c in CHOICES})
    return extract_choice(gen, [row[c] for c in CHOICES])

# -------------------------
# 6) Prompt Formatters
# -------------------------
# CHANGED: SYSTEM_PROMPT (single string) -> SYSTEM_PROMPTS (dict keyed by subject)
SYSTEM_PROMPTS = {
    "clinical_psychologist": (
        "You are a clinical psychology expert with deep knowledge of "
        "cognitive-behavioral therapy, psychopathology, psychological assessment, "
        "mood disorders, anxiety disorders, personality disorders, and "
        "evidence-based treatment. Answer questions accurately based on your expertise."
    ),
    "speech_pathologist": (
        "You are a speech-language pathology expert with deep knowledge of "
        "aphasia, dysarthria, apraxia of speech, stuttering, voice disorders, "
        "dysphagia, hearing disorders, child language development, augmentative "
        "communication, and clinical assessment tools. "
        "Answer questions accurately based on your expertise."
    ),
    "biomedical_engineer": (
        "You are a biomedical engineering expert with deep knowledge of "
        "medical device design, biomechanics, biomaterials, medical imaging, "
        "physiological signal processing, biosensors, tissue engineering, "
        "clinical instrumentation, and regulatory/safety standards for medical "
        "devices. Answer questions accurately based on your expertise."
    ),
    "occupational_therapist": (
        "You are an occupational therapy expert with deep knowledge of "
        "functional assessment, activities of daily living (ADLs) and "
        "instrumental activities of daily living (IADLs), rehabilitation "
        "and habilitation across the lifespan, assistive technology and "
        "adaptive equipment, sensory integration, splinting and positioning, "
        "ergonomics, pediatric and geriatric OT practice, and evidence-based "
        "intervention planning. Answer questions accurately based on your expertise."
    ),
}

ANSWER_FORMAT_INSTRUCTION = (
    "\nAnalyze each option, then state your final answer on the last line "
    "in this exact format:\nAnswer: X\n"
)

def _format_mcq_block(row):
    s = f"Question: {row['question']}\n\n"
    for c in CHOICES:
        s += f"{c}. {row[c]}\n"
    return s

# CHANGED: added `subject` param, uses SYSTEM_PROMPTS[subject]
def format_example_no_rag(row, subject):
    s = (
        f"{SYSTEM_PROMPTS[subject]}\n\n"
        "Answer the following multiple-choice question. "
        "Think through each option carefully, then provide your final answer.\n\n"
        + _format_mcq_block(row)
        + ANSWER_FORMAT_INSTRUCTION
    )
    return s

# CHANGED: added `subject` param, uses SYSTEM_PROMPTS[subject]
def format_example_with_rag(row, passages, subject):
    s = (
        f"{SYSTEM_PROMPTS[subject]}\n\n"
        "Use the reference information below to help answer the "
        "multiple-choice question. If the references don't contain "
        "the answer, use your own knowledge.\n\n"
    )
    if passages:
        s += "=== REFERENCE INFORMATION ===\n"
        for i, p in enumerate(passages, 1):
            s += f"[{i}] {p}\n\n"
        s += "=== END REFERENCES ===\n\n"
    s += _format_mcq_block(row)
    s += (
        "\nAnalyze each option using the references and your expertise, "
        "then state your final answer on the last line in this exact format:\n"
        "Answer: X\n"
    )
    return s

# CHANGED: added `subject` param, uses SYSTEM_PROMPTS[subject]
def format_example_rac(row, exemplars, passages=None, subject=None):
    """RAC prompt: retrieved worked examples (Q + correct answer [+ rationale])
    as few-shot demonstrations, optionally combined with RAG passages.
    Mirrors the paper's balanced few-shot construction (Sec. 3.2)."""
    s = f"{SYSTEM_PROMPTS[subject]}\n\n"

    if passages:
        s += "=== REFERENCE INFORMATION ===\n"
        for i, p in enumerate(passages, 1):
            s += f"[{i}] {p}\n\n"
        s += "=== END REFERENCES ===\n\n"

    if exemplars:
        s += (
            "Below are worked examples of similar questions with their "
            "correct answers. Use them to calibrate your reasoning and "
            "output format, then answer the new question in the same way.\n\n"
        )
        s += "=== WORKED EXAMPLES ===\n"
        for i, ex in enumerate(exemplars, 1):
            s += f"Example {i}:\n"
            s += _format_mcq_block(ex)
            if Config.RAC_INCLUDE_RATIONALE:
                rationale = str(ex.get("exp0", "") or "").strip()
                if rationale:
                    s += f"Rationale: {_truncate_words(rationale, Config.RAC_MAX_EXEMPLAR_WORDS)}\n"
            s += f"Answer: {ex['answer']}\n\n"
        s += "=== END WORKED EXAMPLES ===\n\n"

    s += "Now answer this question:\n\n"
    s += _format_mcq_block(row)
    s += (
        "\nAnalyze each option, following the style of the worked examples "
        "above, then state your final answer on the last line in this exact "
        "format:\nAnswer: X\n"
    )
    return s

def _truncate_words(text, max_words):
    w = text.split()
    return text if len(w) <= max_words else " ".join(w[:max_words]) + "..."

# -------------------------
# 7) Passage KB Load (for RAG / RAC passage grounding)
# -------------------------
class KB:
    def __init__(self, kb_dir):
        with open(os.path.join(kb_dir, "chunks_df.pkl"), "rb") as f:
            df = pickle.load(f)
        self.passages = (
            df["content"] if "content" in df.columns else df["text"]
        ).fillna("").astype(str).tolist()
        self.categories = (
            df["category"].fillna("unknown").astype(str).tolist()
            if "category" in df.columns else ["unknown"] * len(self.passages)
        )
        self.faiss_index = faiss.read_index(os.path.join(kb_dir, "faiss_index.index"))
        with open(os.path.join(kb_dir, "bm25_index.pkl"), "rb") as f:
            self.bm25 = pickle.load(f)
        tok_path = os.path.join(kb_dir, "bm25_tokenizer.pkl")
        if os.path.exists(tok_path):
            with open(tok_path, "rb") as f:
                self.bm25_type = pickle.load(f)
            print("BM25 tokenizer: IMPROVED (stopwords + punctuation)")
        else:
            self.bm25_type = "simple"
            print("BM25 tokenizer: simple (legacy)")
        self.safe_n = min(self.faiss_index.ntotal, len(self.passages))
        print(f"KB loaded: {self.safe_n} passages")

print("Loading Knowledge Base...")
kb = KB(Config.KB_DIR)

# -------------------------
# 8) RAG Retrieval Utils
# -------------------------
def truncate_passage(text, max_words=150):
    return _truncate_words(text, max_words)

# CHANGED: extended with speech_pathologist, biomedical_engineer, occupational_therapist
CATEGORY_HINTS = {
    "clinical_psychologist": [
        "psychology", "psychotherapy", "cognitive", "behavioral", "depression",
        "anxiety", "bipolar", "personality disorder", "assessment", "dsm",
        "therapy", "mental health", "psychopathology",
    ],
    "speech_pathologist": [
        "speech", "language", "communication", "aphasia", "dysarthria",
        "apraxia", "fluency", "stuttering", "voice", "phonation",
        "swallowing", "dysphagia", "hearing", "auditory", "vocal",
        "laryngeal", "articulation", "aac",
    ],
    "biomedical_engineer": [
        "biomedical", "medical device", "biomechanics", "biomaterial",
        "imaging", "signal processing", "biosensor", "tissue engineering",
        "prosthetic", "implant", "instrumentation", "physiological",
        "electrode", "biocompatibility", "medical equipment",
    ],
    "occupational_therapist": [
        "occupational therapy", "occupational therapist", "activities of daily living",
        "adl", "iadl", "functional assessment", "rehabilitation", "habilitation",
        "assistive technology", "adaptive equipment", "orthotics", "splinting",
        "sensory integration", "fine motor", "gross motor", "range of motion",
        "ergonomics", "pediatric therapy", "geriatric care", "disability",
        "activity analysis", "occupational profile", "task modification",
    ],
}

def category_match(cat_text, subject):
    c = str(cat_text).lower()
    if "general_medical" in c:
        return True
    if subject in c:
        return True
    return any(k in c for k in CATEGORY_HINTS.get(subject, []))

def rerank_passages(query, passages, reranker, final_k, threshold=0.0):
    if not passages:
        return []
    pairs = [[query, p] for p in passages]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, passages), key=lambda x: x[0], reverse=True)
    return [passage for score, passage in ranked[:final_k] if score >= threshold]

def retrieve_hybrid(question, subject, embedder, reranker):
    q = Config.BGE_QUERY_PREFIX + question
    q_emb = embedder.encode([q], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_emb)
    sem_scores_raw, sem_idx = kb.faiss_index.search(q_emb, min(Config.RETRIEVE_K, kb.safe_n))
    sem_scores_raw, sem_idx = sem_scores_raw[0], sem_idx[0]

    sem_map = {}
    for i, s in zip(sem_idx, sem_scores_raw):
        i = int(i)
        if 0 <= i < kb.safe_n:
            sem_map[i] = max(0.0, min(1.0, (float(s) + 1.0) / 2.0))

    if kb.bm25_type == "improved":
        query_tokens = tokenize_for_bm25(question)
    else:
        query_tokens = question.lower().split()
    kw_scores = kb.bm25.get_scores(query_tokens)
    kw_top_idx = np.argsort(kw_scores)[::-1][:Config.BM25_TOP]

    candidates = set(sem_map.keys()) | set(int(i) for i in kw_top_idx if 0 <= int(i) < kb.safe_n)

    kw_min, kw_max = float(kw_scores.min()), float(kw_scores.max())
    kw_range = kw_max - kw_min

    def norm_kw(x):
        # FIX: guard divide-by-zero when all BM25 scores are identical
        return 0.0 if kw_range == 0 else (float(x) - kw_min) / kw_range

    fused = []
    for i in candidates:
        s = sem_map.get(i, 0.0)
        k = norm_kw(kw_scores[i])
        score = Config.SEM_WEIGHT * s + (1.0 - Config.SEM_WEIGHT) * k
        if category_match(kb.categories[i], subject):
            score += Config.CATEGORY_BONUS
        fused.append((i, score))
    fused.sort(key=lambda x: x[1], reverse=True)

    top_idx = [i for i, _ in fused[:min(Config.RETRIEVE_K, len(fused))]]
    passages = [truncate_passage(kb.passages[i], Config.MAX_PASSAGE_WORDS) for i in top_idx]
    return rerank_passages(question, passages, reranker, Config.FINAL_K, Config.RERANK_THRESHOLD)

# -------------------------
# 7b) RAC Exemplar Pool + Retrieval
# -------------------------
class ExemplarPool:
    """Labeled Q&A pool that RAC retrieves worked examples from.

    Load order (first that exists wins):
      1. EVAL_DATA_PATH/train/{subject}_train.tsv
      2. EVAL_DATA_PATH/dev/{subject}_dev.tsv
      3. Fallback: the test set itself, with leave-one-out exclusion at
         query time so a row can never retrieve itself as an exemplar.
    """

    def __init__(self, subject, test_df, embedder):
        self.subject = subject
        self.is_self_pool = False

        pool_df = None
        for split, fname_suffix in (("train", "train"), ("dev", "dev")):
            fpath = os.path.join(Config.EVAL_DATA_PATH, split, f"{subject}_{fname_suffix}.tsv")
            if os.path.exists(fpath):
                pool_df = pd.read_csv(
                    fpath, sep="\t",
                    names=["question", "A", "B", "C", "D", "exp0", "exp1", "answer"],
                ).dropna(subset=["question"]).reset_index(drop=True)
                print(f"  RAC exemplar pool: {fpath} ({len(pool_df)} examples)")
                break

        if pool_df is None:
            print("  RAC exemplar pool: no train/dev file found -> "
                  "falling back to leave-one-out over the test set itself.")
            pool_df = test_df.copy().reset_index(drop=True)
            self.is_self_pool = True

        pool_df["answer"] = pool_df["answer"].astype(str).str.strip().str.upper()
        self.df = pool_df

        texts = [Config.BGE_QUERY_PREFIX + q for q in self.df["question"].astype(str).tolist()]
        embs = embedder.encode(texts, convert_to_numpy=True, batch_size=64,
                                show_progress_bar=False).astype("float32")
        faiss.normalize_L2(embs)
        self.index = faiss.IndexFlatIP(embs.shape[1])
        self.index.add(embs)
        self.n = len(self.df)

    def retrieve(self, question, embedder, reranker, exclude_row_id=None):
        q = Config.BGE_QUERY_PREFIX + question
        q_emb = embedder.encode([q], convert_to_numpy=True).astype("float32")
        faiss.normalize_L2(q_emb)

        k = min(Config.RAC_EXEMPLAR_RETRIEVE_K + 1, self.n)  # +1 headroom for self-exclusion
        _, idx = self.index.search(q_emb, k)
        idx = [int(i) for i in idx[0] if 0 <= int(i) < self.n]

        if self.is_self_pool and exclude_row_id is not None:
            idx = [i for i in idx if i != exclude_row_id]
        idx = idx[:Config.RAC_EXEMPLAR_RETRIEVE_K]
        if not idx:
            return []

        candidates = self.df.iloc[idx].reset_index(drop=True)
        pairs = [[question, r["question"]] for _, r in candidates.iterrows()]
        scores = reranker.predict(pairs)

        ranked = sorted(
            zip(scores, candidates.to_dict("records")),
            key=lambda x: x[0], reverse=True,
        )
        # FIX: only apply an absolute-score cutoff if one is explicitly set.
        # Cross-encoder logits for question-vs-question pairs are often
        # negative even when genuinely related, so filtering by score >= 0.0
        # was dropping almost every candidate. Default (None) just keeps the
        # top-ranked candidates regardless of sign.
        if Config.RAC_RERANK_THRESHOLD is not None:
            ranked = [(s, r) for s, r in ranked if s >= Config.RAC_RERANK_THRESHOLD]
        if not ranked:
            return []

        # Balanced greedy selection: mirror the paper's "one example per
        # class first" rule (Sec. 3.2) so exemplar answer-letters aren't
        # dominated by one option. OFF by default -- see RAC_BALANCE_ANSWERS
        # comment in Config: forcing letter diversity in an MCQ context can
        # discard more-relevant exemplars in favor of irrelevant ones.
        if not Config.RAC_BALANCE_ANSWERS:
            top = ranked[:Config.RAC_SHOTS]
            return [r for _, r in top]

        chosen, seen_letters = [], set()
        remaining = list(ranked)
        while remaining and len(chosen) < Config.RAC_SHOTS:
            for i, (s, r) in enumerate(remaining):
                if r["answer"] not in seen_letters:
                    chosen.append(r)
                    seen_letters.add(r["answer"])
                    remaining.pop(i)
                    break
            else:
                # all remaining letters already represented -> take best-ranked leftover
                chosen.append(remaining.pop(0)[1])

        return chosen[:Config.RAC_SHOTS]

# -------------------------
# 9) Load Models
# -------------------------
print("\nLoading embedder...")
embedder = SentenceTransformer(Config.EMBEDDER_MODEL, device=device)
print("Loading reranker...")
reranker = CrossEncoder(Config.RERANKER_MODEL, device=device)

print(f"Loading LLM: {Config.LLM_MODEL}...")

# FIX: if a previous attempt in this same session partially loaded a model
# (e.g. errored out mid-load), those tensors can still be pinned in GPU
# memory. Explicitly drop any leftover references and force garbage
# collection before allocating the real model, so a failed prior attempt
# doesn't starve this one of VRAM.
import gc
for _name in ("model", "pipe"):
    if _name in globals():
        del globals()[_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    free_mem, total_mem = torch.cuda.mem_get_info()
    print(f"  GPU memory before LLM load: {free_mem / 1024**3:.2f} GB free / {total_mem / 1024**3:.2f} GB total")

tokenizer = AutoTokenizer.from_pretrained(Config.LLM_MODEL, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

if Config.USE_4BIT and torch.cuda.is_available():
    print("Using 4-bit quantization (fits T4 16GB)...")
    # FIX: device_map="auto" can decide to offload a few modules to CPU/disk
    # even when the quantized model comfortably fits on GPU (a 4-bit 7B model
    # only needs ~5GB), especially after other models (embedder, reranker)
    # have already claimed some VRAM. Pin everything to GPU 0 explicitly
    # instead of letting "auto" guess, and clear any cached/fragmented
    # memory first.
    torch.cuda.empty_cache()
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        Config.LLM_MODEL, quantization_config=bnb_config,
        device_map={"": 0}, trust_remote_code=True,
    )
else:
    print("Loading in full precision...")
    model = AutoModelForCausalLM.from_pretrained(
        Config.LLM_MODEL,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto", trust_remote_code=True,
    )

# FIX: clear the model's default generation_config.max_length so it can't
# collide with max_new_tokens (this was the actual source of the repeated
# "Both `max_new_tokens` and `max_length` seem to have been set" warning).
model.generation_config.max_length = None

pipe = transformers.pipeline("text-generation", model=model, tokenizer=tokenizer, trust_remote_code=True)

gen_kwargs = dict(
    do_sample=False,
    num_beams=1,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
    max_new_tokens=Config.MAX_NEW_TOKENS,
)
print("All models loaded!\n")

# -------------------------
# 10) Generate Response (handles chat vs completion models)
# -------------------------
def generate_response(prompt):
    if hasattr(tokenizer, "chat_template") and tokenizer.chat_template:
        messages = [{"role": "user", "content": prompt}]
        out = pipe(messages, return_full_text=False, **gen_kwargs)
    else:
        out = pipe(prompt, return_full_text=False, **gen_kwargs)
    text = out[0]["generated_text"]
    # FIX: some pipeline configs return the chat-format list instead of a string
    if isinstance(text, list):
        text = text[-1].get("content", "") if text and isinstance(text[-1], dict) else str(text)
    return text

# -------------------------
# 11) Per-arm scoring helper
# -------------------------
def score_row(row, gold, prompt, extra_cols=None):
    resp = generate_response(prompt)
    pred = extract_answer(resp, row)
    correct = int(pred == gold)
    out = {
        "question": row["question"],
        "A": row["A"], "B": row["B"], "C": row["C"], "D": row["D"],
        "exp0": row.get("exp0", ""), "exp1": row.get("exp1", ""),
        "answer": gold,
        "model_response": resp,
        "model_output": pred,
        "correctness": correct,
    }
    if extra_cols:
        out.update(extra_cols)
    return out, correct

# -------------------------
# 12) Evaluation Loop
# -------------------------
start = time.time()
all_nr, all_r, all_rac = [], [], []

for subject in SUBJECTS:
    print(f"\n{'=' * 60}")
    print(f"  EVALUATING: {subject}")
    print(f"{'=' * 60}")

    fpath = os.path.join(Config.EVAL_DATA_PATH, "test", f"{subject}_test.tsv")
    if not os.path.exists(fpath):
        print(f"[WARN] missing: {fpath}")
        continue

    df = pd.read_csv(
        fpath, sep="\t",
        names=["question", "A", "B", "C", "D", "exp0", "exp1", "answer"],
    ).dropna(subset=["question"]).reset_index(drop=True)
    print(f"  {len(df)} questions loaded\n")

    exemplar_pool = None
    if Config.RUN_RAC:
        print("  Building RAC exemplar pool...")
        exemplar_pool = ExemplarPool(subject, df, embedder)

    rows_nr, rows_r, rows_rac = [], [], []
    correct_nr, correct_r, correct_rac = 0, 0, 0

    for idx, row in tqdm(df.iterrows(), total=len(df), desc=subject):
        gold = str(row["answer"]).strip().upper()

        # ---- No-RAG ----
        p_nr = format_example_no_rag(row, subject)
        rec_nr, is_nr = score_row(row, gold, p_nr)
        rows_nr.append(rec_nr)
        correct_nr += is_nr

        # ---- RAG (passages only) ----
        passages = retrieve_hybrid(row["question"], subject, embedder, reranker)
        p_r = format_example_with_rag(row, passages, subject)
        rec_r, is_r = score_row(row, gold, p_r,
                                 {"retrieved_passages": "\n".join(f"- {p}" for p in passages) if passages else "None"})
        rows_r.append(rec_r)
        correct_r += is_r

        # ---- RAC (retrieved worked examples [+ passages]) ----
        if Config.RUN_RAC:
            exemplars = exemplar_pool.retrieve(row["question"], embedder, reranker, exclude_row_id=idx)
            rac_passages = passages if Config.RAC_COMBINE_WITH_PASSAGES else None
            p_rac = format_example_rac(row, exemplars, rac_passages, subject)
            rec_rac, is_rac = score_row(
                row, gold, p_rac,
                {"exemplars_used": "\n".join(f"- Q: {e['question'][:80]}... -> {e['answer']}" for e in exemplars) if exemplars else "None"},
            )
            rows_rac.append(rec_rac)
            correct_rac += is_rac

        if (idx + 1) % 10 == 0:
            msg = (f"  [{idx + 1}/{len(df)}] No-RAG: {correct_nr}/{idx + 1} ({100 * correct_nr / (idx + 1):.1f}%) | "
                   f"RAG: {correct_r}/{idx + 1} ({100 * correct_r / (idx + 1):.1f}%)")
            if Config.RUN_RAC:
                msg += f" | RAC: {correct_rac}/{idx + 1} ({100 * correct_rac / (idx + 1):.1f}%)"
            print(msg)

    dfn = pd.DataFrame(rows_nr)
    dfr = pd.DataFrame(rows_r)
    dfn.to_csv(os.path.join(Config.OUTPUT_DIR, f"{subject}_no_rag.csv"), index=False)
    dfr.to_csv(os.path.join(Config.OUTPUT_DIR, f"{subject}_rag.csv"), index=False)
    acc_nr = 100.0 * dfn["correctness"].mean()
    acc_r = 100.0 * dfr["correctness"].mean()

    print(f"\n  {'=' * 40}")
    print(f"  {subject} RESULTS:")
    print(f"  No-RAG Accuracy: {acc_nr:.2f}% ({correct_nr}/{len(df)})")
    print(f"  RAG Accuracy:    {acc_r:.2f}% ({correct_r}/{len(df)})")
    print(f"  RAG Δ (vs No-RAG): {acc_r - acc_nr:+.2f}%")

    if Config.RUN_RAC:
        dfrac = pd.DataFrame(rows_rac)
        dfrac.to_csv(os.path.join(Config.OUTPUT_DIR, f"{subject}_rac.csv"), index=False)
        acc_rac = 100.0 * dfrac["correctness"].mean()
        print(f"  RAC Accuracy:    {acc_rac:.2f}% ({correct_rac}/{len(df)})")
        print(f"  RAC Δ (vs No-RAG): {acc_rac - acc_nr:+.2f}%")
        print(f"  RAC Δ (vs RAG):    {acc_rac - acc_r:+.2f}%")
        all_rac.append(dfrac)
    print(f"  {'=' * 40}")

    all_nr.append(dfn)
    all_r.append(dfr)

# -------------------------
# 13) Overall Summary
# -------------------------
if all_nr:
    all_nr_df = pd.concat(all_nr, ignore_index=True)
    all_r_df = pd.concat(all_r, ignore_index=True)
    all_nr_df.to_csv(os.path.join(Config.OUTPUT_DIR, "results_all_no_rag.csv"), index=False)
    all_r_df.to_csv(os.path.join(Config.OUTPUT_DIR, "results_all_rag.csv"), index=False)

    print(f"\n{'=' * 60}")
    print("  OVERALL RESULTS")
    print(f"{'=' * 60}")
    print(f"  Model:          {Config.LLM_MODEL}")
    print(f"  4-bit quant:    {Config.USE_4BIT}")
    print(f"  Questions:      {len(all_nr_df)}")
    print(f"  No-RAG:         {100.0 * all_nr_df['correctness'].mean():.2f}%")
    print(f"  RAG:            {100.0 * all_r_df['correctness'].mean():.2f}%")

    merged = all_nr_df[['question', 'answer', 'model_output', 'correctness']].rename(
        columns={'model_output': 'pred_nr', 'correctness': 'correct_nr'}
    )
    merged['pred_r'] = all_r_df['model_output']
    merged['correct_r'] = all_r_df['correctness']

    if Config.RUN_RAC and all_rac:
        all_rac_df = pd.concat(all_rac, ignore_index=True)
        all_rac_df.to_csv(os.path.join(Config.OUTPUT_DIR, "results_all_rac.csv"), index=False)
        print(f"  RAC:            {100.0 * all_rac_df['correctness'].mean():.2f}%")
        merged['pred_rac'] = all_rac_df['model_output']
        merged['correct_rac'] = all_rac_df['correctness']

        print(f"\n  RAC vs No-RAG Δ: {100.0 * (all_rac_df['correctness'].mean() - all_nr_df['correctness'].mean()):+.2f}%")
        print(f"  RAC vs RAG Δ:    {100.0 * (all_rac_df['correctness'].mean() - all_r_df['correctness'].mean()):+.2f}%")

    print(f"  Saved to:       {Config.OUTPUT_DIR}")

    rag_helped = merged[(merged['correct_nr'] == 0) & (merged['correct_r'] == 1)]
    rag_hurt = merged[(merged['correct_nr'] == 1) & (merged['correct_r'] == 0)]
    print(f"\n  RAG helped (No-RAG wrong -> RAG right): {len(rag_helped)} questions")
    print(f"  RAG hurt  (No-RAG right -> RAG wrong): {len(rag_hurt)} questions")

    if Config.RUN_RAC and all_rac:
        rac_helped_vs_rag = merged[(merged['correct_r'] == 0) & (merged['correct_rac'] == 1)]
        rac_hurt_vs_rag = merged[(merged['correct_r'] == 1) & (merged['correct_rac'] == 0)]
        print(f"  RAC helped (RAG wrong -> RAC right): {len(rac_helped_vs_rag)} questions")
        print(f"  RAC hurt  (RAG right -> RAC wrong): {len(rac_hurt_vs_rag)} questions")

        if len(rac_hurt_vs_rag) > 0:
            print("\n  Questions where RAC hurt (vs RAG):")
            for _, r in rac_hurt_vs_rag.head(5).iterrows():
                print(f"    Q: {r['question'][:80]}...")
                print(f"    Gold: {r['answer']} | RAG: {r['pred_r']} | RAC: {r['pred_rac']}")

elapsed = (time.time() - start) / 60
print(f"\n  Total time: {elapsed:.1f} min")
print(f"{'=' * 60}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

# FIX: same VRAM-fragmentation guard as your MCQ eval script — must be set
# before any CUDA context is created (before torch import anywhere), so it
# sits at the very top.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

print(os.path.exists("/content/drive"))
print(os.path.exists("/content/drive/MyDrive"))

# ======================================================================
# EXPLANATION-QUALITY EVALUATION: No-RAG vs RAG vs RAC
# ======================================================================
# This script does NOT re-score MCQ accuracy (you already did that). It
# takes a file where, for every question, you already have the predicted
# answer + generated explanation from all three pipelines, and scores
# ONLY explanation quality, following the MedExQA paper's methodology
# (Kim et al., 2024) as closely as possible:
#   - classification-gated scoring: wrong answer -> Overall score = 0
#     (Sec. 4.4 / Sec. 4.5 of the paper), but BLEU/ROUGE-L/METEOR/
#     BERTScore are still computed for error analysis
#   - automatic metrics compared against BOTH ground-truth explanations,
#     taking the max (better semantic match) per metric, consistent with
#     the paper's two-explanation design (Table 1, Sec. 5.5)
#   - BERTScore uses a SciBERT-family embedding (paper used scibert), not
#     generic bert-base, to better capture medical-domain nuance
#   - an LLM-judge (0-10 per criterion) is added on top of the paper's
#     3-point human-rating scale, because your Step 2 spec asks for eight
#     separate 0-10 dimensions (Medical Correctness, Relevance, etc.),
#     which the paper's own eval doesn't produce automatically. This is
#     a documented ASSUMPTION, not something in the paper itself.
#
# DIRECTORY / CONFIG CONVENTIONS carried over from your MCQ script:
#   - Google Drive mount, Config class, OUTPUT_DIR under MyDrive
#   - EVAL_DATA_PATH for inputs, same style of "if missing, tell the user"
#   - 4-bit quantized local LLM used as the judge, loaded the same way
#     (BitsAndBytesConfig, device_map pinned to GPU 0, cache clearing)
#
# INSTRUCTIONS:
#   1. Put one combined file per your "Input" spec at:
#        {EVAL_DATA_PATH}/combined_pipeline_results.tsv  (or .csv)
#      with columns (case-insensitive, flexible naming — see
#      COLUMN_ALIASES below):
#        question_id, specialty, question, A, B, C, D, correct_answer,
#        explanation_1, explanation_2,
#        norag_pred, norag_explanation,
#        rag_pred, rag_explanation,
#        rac_pred, rac_explanation
#   2. Run this entire cell in Colab. Output lands in
#        /content/drive/MyDrive/Explanation_Evaluation/
#      exactly matching the folder structure you specified.
# ======================================================================

# -------------------------
# 1) Install
# -------------------------
import subprocess, sys
packages = [
    "transformers>=4.40.0",
    "accelerate>=0.28.0",
    "bitsandbytes>=0.43.0",
    "sacrebleu",
    "rouge-score",
    "nltk",
    "bert-score",
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "tqdm",
]
for p in packages:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])
    except Exception as e:
        print(f"[WARN] {p}: {e}")

# -------------------------
# 2) Imports
# -------------------------
import re
import json
import time
import contextlib
import warnings
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

import sacrebleu
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn

import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from transformers.trainer_utils import set_seed

warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()
import logging as _logging
_logging.getLogger("transformers").setLevel(_logging.ERROR)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

PIPELINES = ["No-RAG", "RAG", "RAC"]


# -------------------------
# 3) Config
# -------------------------
class Config:
    # ===== JUDGE MODEL =====
    JUDGE_MODEL = "Qwen/Qwen2.5-7B-Instruct"   # swap for a larger judge if you have the VRAM
    USE_4BIT = True
    JUDGE_MAX_NEW_TOKENS = 700
    JUDGE_TEMPERATURE = 0.0   # deterministic scoring

    # ===== PATHS (same convention as your MCQ eval script) =====
    EVAL_DATA_PATH = "/content/drive/MyDrive/eval_data"
    INPUT_FILE_CANDIDATES = [
        "combined_pipeline_results.tsv",
        "combined_pipeline_results.csv",
    ]
    # FIX: your MCQ eval script's own OUTPUT_DIR (Config.OUTPUT_DIR there ==
    # RESULTS_DIR here) already contains everything needed to build the
    # combined file — {subject}_no_rag.csv / _rag.csv / _rac.csv, each with
    # question, A-D, exp0/exp1 (ground-truth explanations), answer,
    # model_output (predicted letter), model_response (full explanation).
    # If no manually-built combined file is found, this script now builds
    # one automatically from those files instead of erroring out.
    RESULTS_DIR = "/content/drive/MyDrive/results_improved"
    OUTPUT_DIR = "/content/drive/MyDrive/Explanation_Evaluation"
    PLOTS_DIR = os.path.join(OUTPUT_DIR, "plots")

    # ===== BERTScore =====
    # FIX: the paper explicitly uses SciBERT embeddings for BERTScore (Sec.
    # 4.4), not generic bert-base-uncased, because generic BERT under-values
    # domain terminology. Kept consistent here.
    BERTSCORE_MODEL = "allenai/scibert_scivocab_uncased"
    BERTSCORE_NUM_LAYERS = 9  # standard for scibert-scivocab in bert_score
    # FIX (OverflowError): scibert's tokenizer_config.json ships with no
    # model_max_length, so HF defaults it to a huge sentinel (~1e30). The
    # Rust tokenizer backend overflows converting that into a C long when
    # enable_truncation() is called. We clamp model_max_length to this value
    # and also hard-clip texts to this many words before scoring, which is
    # both the fix and a speed/memory safeguard.
    BERTSCORE_MAX_LENGTH = 512
    BERTSCORE_MAX_WORDS = 400

    # ===== Thresholds for summary counts (documented assumptions) =====
    # "Hallucination" flag: any explanation scoring below the paper's rubric's
    # own "minor unsupported info" ceiling (i.e. anything not a clean 10).
    HALLUCINATION_FLAG_MAX = 9        # Hallucination_Score <= 9 counts as "has some hallucination"
    CLINICALLY_UNSAFE_MAX = 3         # Hallucination_Score 0-3 per your rubric = clinically unsafe
    INCOMPLETE_COMPLETENESS_MAX = 5   # Completeness <= 5 counts as "incomplete"

    SEED = 1234


set_seed(Config.SEED)
os.makedirs(Config.OUTPUT_DIR, exist_ok=True)
os.makedirs(Config.PLOTS_DIR, exist_ok=True)

# -------------------------
# 4) Load input data
# -------------------------
COLUMN_ALIASES = {
    "question_id": ["question_id", "qid", "id"],
    "specialty": ["specialty", "subject", "category"],
    "question": ["question", "question_text"],
    "A": ["a", "option_a", "choice_a"],
    "B": ["b", "option_b", "choice_b"],
    "C": ["c", "option_c", "choice_c"],
    "D": ["d", "option_d", "choice_d"],
    "correct_answer": ["correct_answer", "answer", "gold", "gold_answer"],
    "explanation_1": ["explanation_1", "ground_truth_explanation_1", "exp0", "reference_explanation_1"],
    "explanation_2": ["explanation_2", "ground_truth_explanation_2", "exp1", "reference_explanation_2"],
    "norag_pred": ["norag_pred", "no_rag_pred", "no-rag_predicted_answer", "norag_predicted_answer"],
    "norag_explanation": ["norag_explanation", "no_rag_explanation", "no-rag_explanation"],
    "rag_pred": ["rag_pred", "rag_predicted_answer"],
    "rag_explanation": ["rag_explanation"],
    "rac_pred": ["rac_pred", "rac_predicted_answer"],
    "rac_explanation": ["rac_explanation"],
}


def _find_input_file():
    for fname in Config.INPUT_FILE_CANDIDATES:
        fpath = os.path.join(Config.EVAL_DATA_PATH, fname)
        if os.path.exists(fpath):
            return fpath
    return None


def _normalize_columns(df):
    lower_map = {c.lower().strip(): c for c in df.columns}
    rename = {}
    for canon, aliases in COLUMN_ALIASES.items():
        for alias in aliases:
            if alias in lower_map:
                rename[lower_map[alias]] = canon
                break
    df = df.rename(columns=rename)
    missing = [c for c in COLUMN_ALIASES if c not in df.columns]
    if missing:
        raise ValueError(
            f"Input file is missing required columns (after alias matching): {missing}. "
            f"Found columns: {list(df.columns)}"
        )
    return df


def _strip_answer_line(text):
    """Removes a trailing standalone 'Answer: X' line from a generated
    response so the judge sees the rationale, not the answer restated.
    Harmless no-op if no such line is present."""
    text = "" if text is None or (isinstance(text, float) and np.isnan(text)) else str(text)
    lines = text.split("\n")
    kept = [
        ln for ln in lines
        if not re.match(r"^\s*\**\s*Answer\s*:\s*\**\s*[A-D]\s*\**\.?\s*$", ln.strip(), re.IGNORECASE)
    ]
    return "\n".join(kept).strip()


def _discover_subjects(results_dir):
    """Finds every {subject}_no_rag.csv in results_dir and returns the
    subject names, so this works whether you ran one subject
    (speech_pathologist) or several."""
    subjects = []
    if not os.path.isdir(results_dir):
        return subjects
    for fname in sorted(os.listdir(results_dir)):
        m = re.match(r"^(.+)_no_rag\.csv$", fname)
        if m and m.group(1) != "results_all":
            subjects.append(m.group(1))
    return subjects


def _build_combined_from_mcq_outputs(results_dir):
    """Auto-builds the combined No-RAG/RAG/RAC explanation-eval input
    directly from your MCQ eval script's own per-subject output CSVs
    (question, A-D, exp0/exp1, answer, model_output, model_response),
    matching rows by position since all three arms iterate the same
    test dataframe in the same row order within that script."""
    subjects = _discover_subjects(results_dir)
    if not subjects:
        return None

    all_rows = []
    for subject in subjects:
        nr_path = os.path.join(results_dir, f"{subject}_no_rag.csv")
        rag_path = os.path.join(results_dir, f"{subject}_rag.csv")
        rac_path = os.path.join(results_dir, f"{subject}_rac.csv")
        if not os.path.exists(nr_path):
            continue
        dfn = pd.read_csv(nr_path)
        dfr = pd.read_csv(rag_path) if os.path.exists(rag_path) else None
        dfrac = pd.read_csv(rac_path) if os.path.exists(rac_path) else None

        n = len(dfn)
        if dfr is not None and len(dfr) != n:
            print(f"[WARN] {subject}: no_rag ({n}) and rag ({len(dfr)}) row counts differ — "
                  "check both files came from the same run.")
        if dfrac is not None and len(dfrac) != n:
            print(f"[WARN] {subject}: no_rag ({n}) and rac ({len(dfrac)}) row counts differ — "
                  "check both files came from the same run.")

        for i in range(n):
            row_nr = dfn.iloc[i]
            row_r = dfr.iloc[i] if (dfr is not None and i < len(dfr)) else None
            row_rac = dfrac.iloc[i] if (dfrac is not None and i < len(dfrac)) else None

            rec = {
                "question_id": f"{subject}_{i}",
                "specialty": subject,
                "question": row_nr["question"],
                "A": row_nr["A"], "B": row_nr["B"], "C": row_nr["C"], "D": row_nr["D"],
                "correct_answer": row_nr["answer"],
                "explanation_1": row_nr.get("exp0", ""),
                "explanation_2": row_nr.get("exp1", ""),
                "norag_pred": row_nr.get("model_output", ""),
                "norag_explanation": _strip_answer_line(row_nr.get("model_response", "")),
                "rag_pred": row_r.get("model_output", "") if row_r is not None else "",
                "rag_explanation": _strip_answer_line(row_r.get("model_response", "")) if row_r is not None else "",
                "rac_pred": row_rac.get("model_output", "") if row_rac is not None else "",
                "rac_explanation": _strip_answer_line(row_rac.get("model_response", "")) if row_rac is not None else "",
            }
            all_rows.append(rec)

    if not all_rows:
        return None
    return pd.DataFrame(all_rows)


input_path = _find_input_file()
if input_path is not None:
    sep = "\t" if input_path.endswith(".tsv") else ","
    df = pd.read_csv(input_path, sep=sep)
    df = _normalize_columns(df)
    print(f"Loaded {len(df)} questions from {input_path}")
else:
    print(f"[INFO] No manually-built combined file found in {Config.EVAL_DATA_PATH} "
          f"(looked for {Config.INPUT_FILE_CANDIDATES}). "
          f"Attempting to auto-build one from your MCQ eval script's outputs in "
          f"{Config.RESULTS_DIR} instead...")
    df = _build_combined_from_mcq_outputs(Config.RESULTS_DIR)
    if df is None:
        raise FileNotFoundError(
            f"[WARN] Could not find a combined file in {Config.EVAL_DATA_PATH}, and no "
            f"{{subject}}_no_rag.csv files were found in {Config.RESULTS_DIR} either. "
            "Either drop a combined_pipeline_results.csv/.tsv into EVAL_DATA_PATH, or "
            "point Config.RESULTS_DIR at the folder containing your MCQ script's "
            "{subject}_no_rag.csv / _rag.csv / _rac.csv files."
        )
    os.makedirs(Config.EVAL_DATA_PATH, exist_ok=True)
    auto_path = os.path.join(Config.EVAL_DATA_PATH, "combined_pipeline_results.csv")
    df.to_csv(auto_path, index=False)
    print(f"[INFO] Auto-built combined file from MCQ outputs -> saved to {auto_path} "
          f"({len(df)} rows) for reuse/inspection.")

df["correct_answer"] = df["correct_answer"].astype(str).str.strip().str.upper()

# -------------------------
# 5) Automatic metrics (BLEU / ROUGE-L / METEOR / BERTScore, max over 2 refs)
# -------------------------
_rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)


def _safe_text(x):
    return "" if (x is None or (isinstance(x, float) and np.isnan(x))) else str(x)


def bleu_max(candidate, ref1, ref2):
    candidate, ref1, ref2 = _safe_text(candidate), _safe_text(ref1), _safe_text(ref2)
    if not candidate.strip():
        return 0.0
    scores = []
    for ref in (ref1, ref2):
        if ref.strip():
            scores.append(sacrebleu.sentence_bleu(candidate, [ref]).score)
    return max(scores) if scores else 0.0


def rouge_l_max(candidate, ref1, ref2):
    candidate, ref1, ref2 = _safe_text(candidate), _safe_text(ref1), _safe_text(ref2)
    if not candidate.strip():
        return 0.0
    scores = []
    for ref in (ref1, ref2):
        if ref.strip():
            scores.append(_rouge.score(ref, candidate)["rougeL"].fmeasure * 100)
    return max(scores) if scores else 0.0


def meteor_max(candidate, ref1, ref2):
    candidate, ref1, ref2 = _safe_text(candidate), _safe_text(ref1), _safe_text(ref2)
    if not candidate.strip():
        return 0.0
    cand_tok = word_tokenize(candidate.lower())
    scores = []
    for ref in (ref1, ref2):
        if ref.strip():
            scores.append(meteor_score([word_tokenize(ref.lower())], cand_tok) * 100)
    return max(scores) if scores else 0.0


@contextlib.contextmanager
def _clamp_tokenizer_max_length(max_len=Config.BERTSCORE_MAX_LENGTH if False else 512):
    """FIX for OverflowError: scibert's tokenizer_config.json has no
    model_max_length, so it defaults to a huge sentinel int (~1e30). The
    Rust tokenizer backend used by newer transformers versions overflows
    when that gets passed into enable_truncation(), even on the 'slow'
    tokenizer path. This patches AutoTokenizer.from_pretrained so any
    tokenizer loaded while this context is active gets a sane
    model_max_length immediately after loading."""
    orig_from_pretrained = AutoTokenizer.from_pretrained.__func__

    def patched(cls, *args, **kwargs):
        tok = orig_from_pretrained(cls, *args, **kwargs)
        if getattr(tok, "model_max_length", 0) > 100000:
            tok.model_max_length = max_len
        return tok

    AutoTokenizer.from_pretrained = classmethod(patched)
    try:
        yield
    finally:
        AutoTokenizer.from_pretrained = classmethod(orig_from_pretrained)


def bertscore_batch_max(candidates, refs1, refs2):
    """Batched for speed: computes BERTScore F1 against ref1 and ref2 for the
    whole dataframe at once, then takes the elementwise max.

    FIX: wrapped in _clamp_tokenizer_max_length to avoid the scibert
    tokenizer OverflowError, and texts are word-clipped before scoring as
    an extra safeguard (also keeps this fast and memory-safe)."""
    def _clip(x):
        x = _safe_text(x)
        words = x.split()
        return " ".join(words[:Config.BERTSCORE_MAX_WORDS]) if words else "N/A"

    cands = [_clip(c) for c in candidates]
    r1 = [_clip(r) for r in refs1]
    r2 = [_clip(r) for r in refs2]

    with _clamp_tokenizer_max_length(Config.BERTSCORE_MAX_LENGTH):
        _, _, f1_a = bert_score_fn(
            cands, r1, model_type=Config.BERTSCORE_MODEL,
            num_layers=Config.BERTSCORE_NUM_LAYERS, lang="en", verbose=False,
        )
        _, _, f1_b = bert_score_fn(
            cands, r2, model_type=Config.BERTSCORE_MODEL,
            num_layers=Config.BERTSCORE_NUM_LAYERS, lang="en", verbose=False,
        )
    return np.maximum(f1_a.numpy(), f1_b.numpy()) * 100


# -------------------------
# 6) Load judge LLM (same loading pattern as your MCQ script)
# -------------------------
print(f"Loading judge LLM: {Config.JUDGE_MODEL}...")
import gc
for _name in ("judge_model", "judge_pipe"):
    if _name in globals():
        del globals()[_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    free_mem, total_mem = torch.cuda.mem_get_info()
    print(f"  GPU memory before judge load: {free_mem / 1024**3:.2f} GB free / {total_mem / 1024**3:.2f} GB total")

judge_tokenizer = AutoTokenizer.from_pretrained(Config.JUDGE_MODEL, trust_remote_code=True)
if judge_tokenizer.pad_token_id is None:
    judge_tokenizer.pad_token = judge_tokenizer.eos_token
    judge_tokenizer.pad_token_id = judge_tokenizer.eos_token_id

if Config.USE_4BIT and torch.cuda.is_available():
    torch.cuda.empty_cache()
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    judge_model = AutoModelForCausalLM.from_pretrained(
        Config.JUDGE_MODEL, quantization_config=bnb_config,
        device_map={"": 0}, trust_remote_code=True,
    )
else:
    judge_model = AutoModelForCausalLM.from_pretrained(
        Config.JUDGE_MODEL,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto", trust_remote_code=True,
    )

judge_model.generation_config.max_length = None
judge_pipe = transformers.pipeline("text-generation", model=judge_model, tokenizer=judge_tokenizer, trust_remote_code=True)
print("Judge LLM loaded!\n")

JUDGE_GEN_KWARGS = dict(
    do_sample=False,
    num_beams=1,
    eos_token_id=judge_tokenizer.eos_token_id,
    pad_token_id=judge_tokenizer.eos_token_id,
    max_new_tokens=Config.JUDGE_MAX_NEW_TOKENS,
)

# -------------------------
# 7) LLM-judge prompt + call + robust JSON parsing
# -------------------------
JUDGE_SYSTEM = (
    "You are a rigorous medical education expert grading how well a model's "
    "explanation justifies the correct answer to a multiple-choice medical "
    "question. Score strictly and consistently."
)

JUDGE_CRITERIA = [
    "Medical_Correctness", "Relevance", "Completeness", "Consistency",
    "Clarity", "Conciseness", "Medical_Terminology", "Hallucination",
]


def build_judge_prompt(row, explanation):
    return f"""{JUDGE_SYSTEM}

Question: {row['question']}
A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
Correct Answer: {row['correct_answer']}

Reference Explanation 1: {row['explanation_1']}

Reference Explanation 2: {row['explanation_2']}

Model's Explanation to grade: {explanation}

Score the model's explanation on each of these 8 criteria, 0-10 (integers or
one decimal place):
1. Medical_Correctness - scientific correctness, clinical reasoning, no factual errors
2. Relevance - directly explains why the correct option is correct, no unrelated info
3. Completeness - covers all important medical concepts, no missing key reasoning
4. Consistency - medical reasoning equivalent to BOTH reference explanations (wording may differ)
5. Clarity - easy to understand, logical flow, well organized
6. Conciseness - no repetition, no unnecessary sentences
7. Medical_Terminology - correct, appropriate technical language
8. Hallucination - 10 = no hallucination, 7-9 = minor unsupported info,
   4-6 = moderate hallucination, 0-3 = serious medically incorrect statements

Respond with ONLY a single JSON object, no other text, in exactly this form:
{{"Medical_Correctness": <num>, "Relevance": <num>, "Completeness": <num>,
"Consistency": <num>, "Clarity": <num>, "Conciseness": <num>,
"Medical_Terminology": <num>, "Hallucination": <num>,
"Comments": "<one short paragraph covering strengths, weaknesses, missing
reasoning, hallucinations, clinical usefulness, and a suggested improvement>"}}
"""


def _extract_json(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    blob = match.group(0)
    try:
        return json.loads(blob)
    except Exception:
        blob2 = re.sub(r",\s*}", "}", blob)
        try:
            return json.loads(blob2)
        except Exception:
            return None


def judge_generate(prompt):
    if hasattr(judge_tokenizer, "chat_template") and judge_tokenizer.chat_template:
        messages = [{"role": "user", "content": prompt}]
        out = judge_pipe(messages, return_full_text=False, **JUDGE_GEN_KWARGS)
    else:
        out = judge_pipe(prompt, return_full_text=False, **JUDGE_GEN_KWARGS)
    text = out[0]["generated_text"]
    if isinstance(text, list):
        text = text[-1].get("content", "") if text and isinstance(text[-1], dict) else str(text)
    return text


def judge_explanation(row, explanation):
    """Returns a dict of the 8 criteria (0-10) + Comments. Falls back to
    all-zero scores + an error comment if the judge output can't be parsed,
    so the pipeline never silently drops a row."""
    explanation = _safe_text(explanation)
    if not explanation.strip():
        return {**{k: 0.0 for k in JUDGE_CRITERIA}, "Comments": "No explanation was generated."}

    prompt = build_judge_prompt(row, explanation)
    raw = judge_generate(prompt)
    parsed = _extract_json(raw)
    if parsed is None:
        return {**{k: 0.0 for k in JUDGE_CRITERIA},
                "Comments": f"[JUDGE PARSE FAILURE] raw output: {raw[:300]}"}

    result = {}
    for k in JUDGE_CRITERIA:
        try:
            v = float(parsed.get(k, 0.0))
        except Exception:
            v = 0.0
        result[k] = max(0.0, min(10.0, v))
    result["Comments"] = str(parsed.get("Comments", "")).strip()
    return result


# -------------------------
# 8) Per-row, per-pipeline scoring
# -------------------------
PRED_COL = {"No-RAG": "norag_pred", "RAG": "rag_pred", "RAC": "rac_pred"}
EXP_COL = {"No-RAG": "norag_explanation", "RAG": "rag_explanation", "RAC": "rac_explanation"}


def score_pipeline_row(row, pipeline):
    pred = str(row[PRED_COL[pipeline]]).strip().upper()
    explanation = row[EXP_COL[pipeline]]
    gold = row["correct_answer"]
    is_correct = int(pred == gold)

    bleu = bleu_max(explanation, row["explanation_1"], row["explanation_2"])
    rouge_l = rouge_l_max(explanation, row["explanation_1"], row["explanation_2"])
    meteor = meteor_max(explanation, row["explanation_1"], row["explanation_2"])
    # BERTScore is computed in a separate batched pass for speed (see main loop)

    if is_correct:
        judged = judge_explanation(row, explanation)
        overall = float(np.mean([judged[k] for k in JUDGE_CRITERIA]))
    else:
        # STEP 1 rule: wrong answer -> Overall Explanation Score = 0,
        # but we still record judge sub-scores for error analysis rather
        # than leaving them blank, since Step 4/7 error-analysis fields
        # (hallucinations, incomplete explanations, etc.) need them.
        judged = judge_explanation(row, explanation)
        overall = 0.0

    return {
        "Question_ID": row["question_id"],
        "Specialty": row["specialty"],
        "Pipeline": pipeline,
        "Correct_Answer": gold,
        "Predicted_Answer": pred,
        "Answer_Correct": bool(is_correct),
        "Medical_Correctness": judged["Medical_Correctness"],
        "Relevance": judged["Relevance"],
        "Completeness": judged["Completeness"],
        "Consistency": judged["Consistency"],
        "Clarity": judged["Clarity"],
        "Conciseness": judged["Conciseness"],
        "Medical_Terminology": judged["Medical_Terminology"],
        "Hallucination": judged["Hallucination"],
        "BLEU": bleu,
        "ROUGE_L": rouge_l,
        "METEOR": meteor,
        "BERTScore": np.nan,  # filled in after batched computation below
        "Overall_Explanation_Score": overall,
        "Comments": judged["Comments"],
    }


# -------------------------
# 9) Main evaluation loop
# -------------------------
start = time.time()
detailed_rows = []
for pipeline in PIPELINES:
    print(f"\n{'=' * 60}\n  SCORING PIPELINE: {pipeline}\n{'=' * 60}")
    for _, row in tqdm(df.iterrows(), total=len(df), desc=pipeline):
        detailed_rows.append(score_pipeline_row(row, pipeline))

detailed_df = pd.DataFrame(detailed_rows)

# Batched BERTScore per pipeline (much faster than row-by-row)
print("\nComputing BERTScore (batched)...")
for pipeline in PIPELINES:
    mask = detailed_df["Pipeline"] == pipeline
    sub = df.copy()
    sub["Question_ID"] = sub["question_id"]
    merged = detailed_df[mask].merge(
        sub[["question_id", "explanation_1", "explanation_2"]],
        left_on="Question_ID", right_on="question_id", how="left",
    )
    cands = [
        _safe_text(row[EXP_COL[pipeline]])
        for _, row in df.iterrows()
    ]
    bscores = bertscore_batch_max(cands, df["explanation_1"].tolist(), df["explanation_2"].tolist())
    detailed_df.loc[mask, "BERTScore"] = bscores
    print(f"  {pipeline}: done")

detailed_out = os.path.join(Config.OUTPUT_DIR, "detailed_scores.csv")
detailed_df.to_csv(detailed_out, index=False)
print(f"Saved: {detailed_out}")

# -------------------------
# 10) Summary statistics
# -------------------------
def summarize(group):
    n = len(group)
    n_wrong = int((~group["Answer_Correct"]).sum())
    n_halluc = int((group["Hallucination"] <= Config.HALLUCINATION_FLAG_MAX).sum())
    n_incomplete = int((group["Completeness"] <= Config.INCOMPLETE_COMPLETENESS_MAX).sum())
    n_unsafe = int((group["Hallucination"] <= Config.CLINICALLY_UNSAFE_MAX).sum())
    return pd.Series({
        "N": n,
        "Avg_Medical_Correctness": group["Medical_Correctness"].mean(),
        "Avg_Relevance": group["Relevance"].mean(),
        "Avg_Completeness": group["Completeness"].mean(),
        "Avg_Consistency": group["Consistency"].mean(),
        "Avg_Clarity": group["Clarity"].mean(),
        "Avg_Conciseness": group["Conciseness"].mean(),
        "Avg_Medical_Terminology": group["Medical_Terminology"].mean(),
        "Avg_Hallucination": group["Hallucination"].mean(),
        "Avg_BLEU": group["BLEU"].mean(),
        "Avg_ROUGE_L": group["ROUGE_L"].mean(),
        "Avg_METEOR": group["METEOR"].mean(),
        "Avg_BERTScore": group["BERTScore"].mean(),
        "Avg_Overall_Explanation_Score": group["Overall_Explanation_Score"].mean(),
        "Median_Overall_Score": group["Overall_Explanation_Score"].median(),
        "Std_Overall_Score": group["Overall_Explanation_Score"].std(),
        "Min_Overall_Score": group["Overall_Explanation_Score"].min(),
        "Max_Overall_Score": group["Overall_Explanation_Score"].max(),
        "Num_Wrong_Answers": n_wrong,
        "Num_Hallucinations": n_halluc,
        "Num_Incomplete_Explanations": n_incomplete,
        "Num_Clinically_Unsafe_Explanations": n_unsafe,
    })


summary_df = detailed_df.groupby("Pipeline").apply(summarize).reset_index()
summary_df = summary_df.set_index("Pipeline").loc[PIPELINES].reset_index()
summary_out = os.path.join(Config.OUTPUT_DIR, "summary_statistics.csv")
summary_df.to_csv(summary_out, index=False)
print(f"Saved: {summary_out}")

# -------------------------
# 11) Pipeline comparison table
# -------------------------
def get_metric(pipeline, col):
    return float(summary_df.loc[summary_df["Pipeline"] == pipeline, col].iloc[0])


COMPARISON_METRICS = {
    "Overall_Explanation_Score": "Avg_Overall_Explanation_Score",
    "Medical_Correctness": "Avg_Medical_Correctness",
    "Completeness": "Avg_Completeness",
    "Hallucination_Rate": "Avg_Hallucination",   # higher = fewer hallucinations
    "BLEU": "Avg_BLEU",
    "ROUGE_L": "Avg_ROUGE_L",
    "METEOR": "Avg_METEOR",
    "BERTScore": "Avg_BERTScore",
    "Clinical_Usefulness": "Avg_Relevance",  # proxy: relevance to clinical justification
}

comparisons = [("No-RAG", "RAG"), ("RAG", "RAC"), ("No-RAG", "RAC")]
comp_rows = []
for base, comp in comparisons:
    row = {"Comparison": f"{comp} vs {base}"}
    for label, col in COMPARISON_METRICS.items():
        diff = get_metric(comp, col) - get_metric(base, col)
        row[f"Diff_{label}"] = diff
        row[f"Best_{label}"] = comp if diff > 0 else (base if diff < 0 else "Tie")
    comp_rows.append(row)

comparison_df = pd.DataFrame(comp_rows)
comparison_out = os.path.join(Config.OUTPUT_DIR, "comparison_table.csv")
comparison_df.to_csv(comparison_out, index=False)
print(f"Saved: {comparison_out}")

# -------------------------
# 12) Visualizations
# -------------------------
sns.set_theme(style="whitegrid")
PALETTE = {"No-RAG": "#9E9E9E", "RAG": "#4C72B0", "RAC": "#55A868"}


def savefig(name):
    path = os.path.join(Config.PLOTS_DIR, name)
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()
    print(f"Saved: {path}")


# 1. Overall score bar chart
plt.figure(figsize=(6, 5))
sns.barplot(data=summary_df, x="Pipeline", y="Avg_Overall_Explanation_Score",
            order=PIPELINES, palette=PALETTE)
plt.title("Average Overall Explanation Score by Pipeline")
plt.ylabel("Overall Explanation Score (0-10)")
savefig("overall_bar.png")

# 2. Radar chart
radar_metrics = ["Avg_Medical_Correctness", "Avg_Relevance", "Avg_Completeness",
                  "Avg_Consistency", "Avg_Clarity", "Avg_Conciseness",
                  "Avg_Medical_Terminology", "Avg_Hallucination"]
radar_labels = [m.replace("Avg_", "") for m in radar_metrics]
angles = np.linspace(0, 2 * np.pi, len(radar_metrics), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
for pipeline in PIPELINES:
    vals = summary_df.loc[summary_df["Pipeline"] == pipeline, radar_metrics].values.flatten().tolist()
    vals += vals[:1]
    ax.plot(angles, vals, label=pipeline, color=PALETTE[pipeline], linewidth=2)
    ax.fill(angles, vals, color=PALETTE[pipeline], alpha=0.08)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, fontsize=9)
ax.set_ylim(0, 10)
ax.set_title("Explanation Quality Radar (0-10 scale)")
ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.1))
savefig("radar.png")

# 3. Metric-wise comparison chart (automatic metrics)
auto_metrics = ["Avg_BLEU", "Avg_ROUGE_L", "Avg_METEOR", "Avg_BERTScore"]
melted = summary_df.melt(id_vars="Pipeline", value_vars=auto_metrics,
                          var_name="Metric", value_name="Score")
melted["Metric"] = melted["Metric"].str.replace("Avg_", "")
plt.figure(figsize=(8, 5))
sns.barplot(data=melted, x="Metric", y="Score", hue="Pipeline",
            hue_order=PIPELINES, palette=PALETTE)
plt.title("Automatic Metric Comparison Across Pipelines")
savefig("metrics.png")

# 4. Box plot of overall scores
plt.figure(figsize=(6, 5))
sns.boxplot(data=detailed_df, x="Pipeline", y="Overall_Explanation_Score",
            order=PIPELINES, palette=PALETTE)
plt.title("Distribution of Overall Explanation Scores")
savefig("boxplot.png")

# 5. Histogram of overall scores
plt.figure(figsize=(7, 5))
for pipeline in PIPELINES:
    sns.histplot(detailed_df.loc[detailed_df["Pipeline"] == pipeline, "Overall_Explanation_Score"],
                 label=pipeline, color=PALETTE[pipeline], kde=True, stat="density", alpha=0.4)
plt.title("Histogram of Overall Explanation Scores")
plt.legend()
savefig("histogram.png")

# 6. Hallucination comparison
plt.figure(figsize=(6, 5))
sns.barplot(data=summary_df, x="Pipeline", y="Avg_Hallucination",
            order=PIPELINES, palette=PALETTE)
plt.title("Average Hallucination Score by Pipeline (10 = none)")
savefig("hallucination.png")

# 7. Medical correctness comparison
plt.figure(figsize=(6, 5))
sns.barplot(data=summary_df, x="Pipeline", y="Avg_Medical_Correctness",
            order=PIPELINES, palette=PALETTE)
plt.title("Average Medical Correctness by Pipeline")
savefig("medical_correctness.png")

# -------------------------
# 13) Final research report
# -------------------------
ranking = summary_df.sort_values("Avg_Overall_Explanation_Score", ascending=False)["Pipeline"].tolist()

def d(base, comp, col):
    return get_metric(comp, col) - get_metric(base, col)

report = f"""# Explanation Quality Evaluation Report: No-RAG vs RAG vs RAC

## 1. Overall Ranking

Ranked by average Overall Explanation Score (highest to lowest):
{", ".join(f"**{p}**" for p in ranking)}

| Pipeline | Avg Overall Score | Median | Std | Min | Max |
|---|---|---|---|---|---|
"""
for p in PIPELINES:
    r = summary_df[summary_df["Pipeline"] == p].iloc[0]
    report += f"| {p} | {r['Avg_Overall_Explanation_Score']:.2f} | {r['Median_Overall_Score']:.2f} | {r['Std_Overall_Score']:.2f} | {r['Min_Overall_Score']:.2f} | {r['Max_Overall_Score']:.2f} |\n"

report += f"""
## 2. Summary of Automatic Metrics

| Pipeline | BLEU | ROUGE-L | METEOR | BERTScore |
|---|---|---|---|---|
"""
for p in PIPELINES:
    r = summary_df[summary_df["Pipeline"] == p].iloc[0]
    report += f"| {p} | {r['Avg_BLEU']:.2f} | {r['Avg_ROUGE_L']:.2f} | {r['Avg_METEOR']:.2f} | {r['Avg_BERTScore']:.2f} |\n"

report += f"""
## 3. Summary of Qualitative (LLM-Judge) Evaluation

| Pipeline | Medical Correctness | Relevance | Completeness | Consistency | Clarity | Conciseness | Terminology | Hallucination (10=none) |
|---|---|---|---|---|---|---|---|---|
"""
for p in PIPELINES:
    r = summary_df[summary_df["Pipeline"] == p].iloc[0]
    report += (f"| {p} | {r['Avg_Medical_Correctness']:.2f} | {r['Avg_Relevance']:.2f} | "
               f"{r['Avg_Completeness']:.2f} | {r['Avg_Consistency']:.2f} | {r['Avg_Clarity']:.2f} | "
               f"{r['Avg_Conciseness']:.2f} | {r['Avg_Medical_Terminology']:.2f} | {r['Avg_Hallucination']:.2f} |\n")

report += f"""
## 4. Key Improvements: No-RAG → RAG

- Overall Explanation Score change: {d('No-RAG', 'RAG', 'Avg_Overall_Explanation_Score'):+.2f}
- Medical Correctness change: {d('No-RAG', 'RAG', 'Avg_Medical_Correctness'):+.2f}
- Completeness change: {d('No-RAG', 'RAG', 'Avg_Completeness'):+.2f}
- Hallucination score change (higher is better): {d('No-RAG', 'RAG', 'Avg_Hallucination'):+.2f}
- BERTScore change: {d('No-RAG', 'RAG', 'Avg_BERTScore'):+.2f}

## 5. Key Improvements: RAG → RAC

- Overall Explanation Score change: {d('RAG', 'RAC', 'Avg_Overall_Explanation_Score'):+.2f}
- Medical Correctness change: {d('RAG', 'RAC', 'Avg_Medical_Correctness'):+.2f}
- Completeness change: {d('RAG', 'RAC', 'Avg_Completeness'):+.2f}
- Hallucination score change (higher is better): {d('RAG', 'RAC', 'Avg_Hallucination'):+.2f}
- BERTScore change: {d('RAG', 'RAC', 'Avg_BERTScore'):+.2f}

## 6. Error Analysis

| Pipeline | Wrong Answers | Hallucinations | Incomplete Explanations | Clinically Unsafe |
|---|---|---|---|---|
"""
for p in PIPELINES:
    r = summary_df[summary_df["Pipeline"] == p].iloc[0]
    report += (f"| {p} | {int(r['Num_Wrong_Answers'])} | {int(r['Num_Hallucinations'])} | "
               f"{int(r['Num_Incomplete_Explanations'])} | {int(r['Num_Clinically_Unsafe_Explanations'])} |\n")

worst_halluc_examples = detailed_df.sort_values("Hallucination").head(5)[
    ["Question_ID", "Pipeline", "Hallucination", "Comments"]
]
report += "\n### Representative low-hallucination-score (worst) cases\n\n"
for _, r in worst_halluc_examples.iterrows():
    report += f"- **{r['Pipeline']} / Q{r['Question_ID']}** (Hallucination={r['Hallucination']:.1f}): {r['Comments'][:220]}\n"

report += f"""
## 7. Final Conclusion

Across {len(df)} questions scored under all three pipelines, **{ranking[0]}**
produced the highest average Overall Explanation Score
({get_metric(ranking[0], 'Avg_Overall_Explanation_Score'):.2f}/10), following the
MedExQA classification-gated scoring rule (an incorrect predicted answer forces
Overall Explanation Score = 0, matching the paper's Sec. 4.4-4.5 methodology).
RAG produced a {d('No-RAG', 'RAG', 'Avg_Overall_Explanation_Score'):+.2f}-point
average change over No-RAG, and RAC produced a further
{d('RAG', 'RAC', 'Avg_Overall_Explanation_Score'):+.2f}-point change over RAG,
consistent with the general pattern in the RAG/RAC literature that retrieving
solved, labeled exemplars for few-shot conditioning improves not just
answer accuracy but the reasoning quality behind it.

### Documented assumptions
- LLM-judge scores (Medical_Correctness ... Hallucination) are produced by a
  local {Config.JUDGE_MODEL} model prompted with a fixed rubric and both
  reference explanations; this supplements, but does not replace, the
  paper's own 3-point human-annotator scale.
- BERTScore uses {Config.BERTSCORE_MODEL} (paper-consistent SciBERT choice),
  BLEU via sacrebleu, ROUGE-L via rouge-score, METEOR via nltk/WordNet.
- "Hallucinations" = Hallucination_Score <= {Config.HALLUCINATION_FLAG_MAX};
  "Clinically Unsafe" = Hallucination_Score <= {Config.CLINICALLY_UNSAFE_MAX}
  (serious medically incorrect statements per the provided rubric);
  "Incomplete" = Completeness <= {Config.INCOMPLETE_COMPLETENESS_MAX}.
  These thresholds are configurable in `Config`.
"""

report_out = os.path.join(Config.OUTPUT_DIR, "evaluation_report.md")
with open(report_out, "w") as f:
    f.write(report)
print(f"Saved: {report_out}")

elapsed = (time.time() - start) / 60
print(f"\nAll done in {elapsed:.1f} min. Output folder: {Config.OUTPUT_DIR}")